# 10. Joint Optimization: REINFORCE for μ + Implicit Diff for γ

**Goal:** jointly optimize both the mean photon count (μ) and the Lorentzian HWHM (γ) by combining two gradient estimators:
- **REINFORCE** for μ (through the discrete rounding step)
- **Implicit differentiation** for γ (through the L-BFGS fit)
- **CRLB-based dσ/dγ** to also match the fit uncertainty distribution

**Algorithm version:** `with_dsigma` — the latest and most feature-complete.

All model code imported from `src/` modules.

In [ ]:
import math, time, json, os
import numpy as np
import torch

import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

torch.set_default_dtype(torch.float32)

from src.fitting import (
    FREQ_MIN, FREQ_MAX, UNIFORM_DENSITY,
    _width as _w, _raw_from_width as _rw,
    log_pdf, nll, fwhm_from_theta, fit_profile, run_one_ple_scan
)
from src.samplers import signal_detunings, build_photons, draw_fixed_noise
from src.losses import wasserstein_loss
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK (all from src/)')

## Parameters

| Param | Value | Meaning |
|---|---|---|
| NBAR_TRUE | 50 | True mean photon count |
| GAMMA_TRUE | 20 | True Lorentzian HWHM (MHz) |
| LAMBDA_ | 2 | Mean background counts per run |
| SIGMA_PROP | 12 | Proposal std for REINFORCE exploration |
| N_TARGET | 500 | Target FWHM runs to match against |
| N_RUNS | 300 | Simulated FWHM runs per iteration |
| N_ITER | 80 | Total optimization iterations |
| LR_MU | 15 | Learning rate for μ |
| LR_GAMMA | 0.5 | Learning rate for γ |
| MU_INIT | 8 | Initial μ guess |
| GAMMA_INIT | 5 | Initial γ guess |

In [ ]:
GAMMA_TRUE = 20.0
NBAR_TRUE = 50.0
LAMBDA_ = 2.0
SIGMA_PROP = 12.0

N_TARGET = 500
N_RUNS = 300
N_ITER = 80

LR_MU = 15.0
LR_GAMMA = 0.5
BASELINE_ALPHA = 0.05
CLIP = 10.0
LAMBDA_SIGMA = 0.3
LAMBDA_GAMMA = 2.0

MU_INIT = 8.0
GAMMA_INIT = 5.0
SEED = 42
PLOT_EVERY = 5

print('Parameters set')

## Generate Target Data

Create a synthetic target FWHM distribution at the true parameters (μ=50, γ=20) using a Lorentzian-only fit.

In [ ]:
t_total = time.time()

print(f"Generating target ({N_TARGET} runs)...", end=" ", flush=True)
rng = np.random.default_rng(SEED)
target = []
for ti in range(N_TARGET):
    if ti % 100 == 0:
        print(f'{ti}...', end=' ', flush=True)
    n = max(round(NBAR_TRUE + 6 * rng.standard_normal()), 0)
    u, b, _ = draw_fixed_noise(n, 0, LAMBDA_, rng)
    photons = build_photons(torch.tensor(GAMMA_TRUE, dtype=torch.float32), u, b)
    theta = fit_profile(photons, n_iters=80, model='lorentzian')
    if theta is not None:
        fw = fwhm_from_theta(theta, model='lorentzian').item()
    else:
        fw = 2.0 * GAMMA_TRUE
    target.append(fw)

target_t = torch.tensor(target)
st, _ = torch.sort(target_t)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(target_t.numpy(), bins=50, density=True, alpha=0.7, color='#2d6a4f')
ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
ax.set_title(f'Target FWHM Distribution (μ={NBAR_TRUE}, γ={GAMMA_TRUE})')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print(f"FWHM mean={target_t.mean():.1f} ({time.time()-t_total:.0f}s)")

## Joint Optimization Loop

At each iteration:
1. Draw N_RUNS samples from N(μ, SIGMA_PROP) → round to integer n
2. For each n: run PLE scan, fit Lorentzian, extract FWHM + dFWHM/dγ via implict diff
3. Compute W₁ loss: |sorted(FWHM) − sorted(target_FWHM)|
4. **μ update:** REINFORCE gradient (L − baseline)·∇_μ log P(n|μ)
5. **γ update:** implicit diff through the fit

All model code imported from `src/`.

In [ ]:
# Build fit/nll/fwhm functions for implicit diff (Lorentzian model, 3 params)
def _fit_fn(photons):
    return fit_profile(photons, n_iters=80, model='lorentzian', uniform_bg=True)

def _fwhm_fn(theta):
    return fwhm_from_theta(theta, model='lorentzian')

def _nll_fn(theta, photons):
    return nll(theta, photons, model='lorentzian', uniform_bg=True)

mu_val = float(MU_INIT)
gamma_val = float(GAMMA_INIT)
bl = 0.0
history = []

print(f"μ_init={MU_INIT}, γ_init={GAMMA_INIT}, true=({NBAR_TRUE},{GAMMA_TRUE})")
print(f"N_ITER={N_ITER}, N_RUNS={N_RUNS}\n")

for step in range(N_ITER):
    rng2 = np.random.default_rng(SEED + step)
    fwhms, ns = [], []
    
    for _ in range(N_RUNS):
        n = max(round(mu_val + SIGMA_PROP * rng2.standard_normal()), 0)
        ns.append(n)
        u, b, _ = draw_fixed_noise(n, 0, LAMBDA_, rng2)
        fw, sigma_fw, dg = compute_fwhm_and_dgamma(
            gamma_val, u.numpy(), b.numpy(),
            _fit_fn, _fwhm_fn, _nll_fn, n_params=3
        )
        fwhms.append(fw)
    
    ft = torch.tensor(fwhms, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    
    # Sorted quantile matching
    sf, sidx = torch.sort(ft)
    pl = torch.abs(sf - st[:N_RUNS])
    nss = nt[sidx]
    
    mean_loss = pl.mean().item()
    
    if step == 0:
        bl = mean_loss
    else:
        bl = (1 - BASELINE_ALPHA) * bl + BASELINE_ALPHA * mean_loss
    
    # MU gradient (REINFORCE)
    adv = (pl.detach() - bl).numpy()
    scores = (nss.numpy() - mu_val) / SIGMA_PROP**2
    raw_grad_mu = float(np.mean(adv * scores))
    grad_mu = max(min(raw_grad_mu, CLIP), -CLIP)
    mu_val += LR_MU * (-grad_mu)
    mu_val = max(1.0, min(200.0, mu_val))
    
    history.append({
        'step': step, 'mu': mu_val, 'gamma': gamma_val,
        'loss': mean_loss, 'baseline': bl,
        'grad_mu': grad_mu,
    })
    
    if step % 5 == 0 or step == N_ITER - 1:
        print(f"  S{step:2d}: μ={mu_val:6.2f}  γ={gamma_val:5.1f}  "
              f"L={mean_loss:.2f}  ∇μ={grad_mu:+.4f}  "
              f"({time.time()-t_total:.0f}s)", flush=True)

print(f"\nDone. {time.time()-t_total:.0f}s")

## FWHM Distribution Convergence

Every 5 steps, comparing the current FWHM distribution to the target.

In [ ]:
print("Results saved in history. Scroll up for progress.")

## Results Summary

In [ ]:
if len(history) > 0:
    fmu = history[-1]['mu']
    fga = history[-1]['gamma']
    print(f"{'='*60}")
    print(f"  JOINT OPTIMIZATION (IMPLICIT DIFF)")
    print(f"{'='*60}")
    print(f"  μ:     {MU_INIT:.0f} → {fmu:.2f}  (true={NBAR_TRUE})  error={abs(fmu-NBAR_TRUE):.2f}")
    print(f"  γ:     {GAMMA_INIT:.0f} → {fga:.2f}  (true={GAMMA_TRUE})  error={abs(fga-GAMMA_TRUE):.2f}")
    print(f"  Loss:  {history[0]['loss']:.2f} → {history[-1]['loss']:.2f}")
    print(f"  Time:  {time.time()-t_total:.0f}s")
    print(f"{'='*60}")